# 01 · Quickstart

Build a five-year income statement with `finmodel` in a few lines.

We will:

1. Declare an `Inputs` dataclass with our assumptions.

2. Subclass `Model` and define each line item as an `@row` method.

3. `calculate()` and render the styled table.

## 1. Imports and assumptions

Inputs are a plain `@dataclass`. Making the model generic over it
(`Model[Inputs]`) gives full autocomplete on `self.inputs`.

In [ ]:
from dataclasses import dataclass

from finmodel import Model, row, PredefinedFormats as F, PredefinedStyles as S



@dataclass

class Inputs:

    starting_revenue: float

    growth_rate: float

    cost_ratio: float

    tax_rate: float

## 2. Define the model

Each row is a method `(self, t)` where `t` is the zero-based period.

Reference the previous period with `self.revenue(t - 1)` and other rows
with `self.<row>(t)` — results are cached automatically.

In [ ]:
class IncomeStatement(Model[Inputs]):

    @row(group="Revenue", format=F.USD)

    def revenue(self, t):

        if t == 0:

            return self.inputs.starting_revenue

        return self.revenue(t - 1) * (1 + self.inputs.growth_rate)



    @row(group="Costs", format=F.USD)

    def operating_costs(self, t):

        return self.revenue(t) * self.inputs.cost_ratio



    @row(group="Earnings", format=F.SUBTOTAL)

    def ebit(self, t):

        return self.revenue(t) - self.operating_costs(t)



    @row(group="Earnings", format=F.USD)

    def tax(self, t):

        return max(self.ebit(t), 0) * self.inputs.tax_rate



    @row(group="Earnings", format=F.TOTAL)

    def net_income(self, t):

        return self.ebit(t) - self.tax(t)



    @row(group="Earnings", format=F.PERCENTAGE)

    def net_margin(self, t):

        return self.net_income(t) / self.revenue(t)

## 3. Run and display

Instantiate with the number of `periods` and our `inputs`, call
`calculate()`, then `show()` to render the styled table inline.

In [ ]:
inputs = Inputs(starting_revenue=1_000, growth_rate=0.12,

                cost_ratio=0.55, tax_rate=0.25)



model = IncomeStatement(periods=5, inputs=inputs, style=S.CORPORATE_BLUE)

model.calculate()

model.show()

## 4. Get the numbers out

Beyond the styled view you can pull raw arrays or a plain DataFrame.

In [ ]:
print("rows:", model.get_rows())

print("net income by year:", model.get_result_data("net_income"))

model.df()

Export a standalone HTML file with `model.to_html("income.html")`.